# Precision at K and Recall at K retrieval evaluation metric implementation
### Specifically, these fall under `predictive metrics`.

## Precision @ K
Precision allows you to guage the system's ability to retrieve relevant context in the top K items. It focuses on `correctness`

It answers the question: "How many of the top k items are relevant?"

Precision is obtained by takeing the number of relevant documents/ K.
It works well where the number of relevant items/ documents per test question or user is large while the user's attention or our general focus is on the first few retrieved items.

Precision is not a good measure of correctness where the number of relevant items for a test question or user is really low. Especially where its lower than K.
In such situations Precision is always `capped`. i.e.,

```python
   #if number of total relevant items is 3 for a given question
   #and  K=10
   #the highest Precision score is:
    precision_score = 3/10 
```
This also shows that Precision is not a good poplutation metric since tests or users with very low relevant items skew the data.

In addition:
Precision doesn't account for ranking of relevant items within the top k items. It only cares about presence of the relevant items but not the `order`.

In [ ]:
#test data loader implementation

import json
from pydantic import BaseModel, Field

class TestQuestion(BaseModel):
    """ A test question containing labeled fields for RAG evaluation."""
    question: str = Field(description="The question being asked.")
    keywords: list[str] = Field(description="The keywords that must appear in the retrieved documents.")
    reference_answer: str  = Field(description="The reference answer for the question.")
    category: str = Field(description="The category of the question (i.e., direct_fact, spanning, comparative, numerical)")

def load_tests() ->list[TestQuestion]:
    """Loads test quesions from JSONL file into the defined format"""
    TEST_FILE = "tests.jsonl"
    tests = []

    with open(TEST_FILE, 'r', encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            tests.append(TestQuestion(**data))
    return tests


In [ ]:
tests = load_tests()
print(tests[0])

In [ ]:
#precision @ k implementation
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv(override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
retriever = Chroma(persist_directory="../vector_db", embedding_function=embeddings).as_retriever()


In [ ]:
#precison = number of relevant docs / k
def calculate_precision(tests: TestQuestion) ->float:
    scores = []
    for test in tests:
        docs = retriever.invoke(test.question)
        key_rel = []
        precision_scores = []
        for keyword in test.keywords:
            key_rel = [1 if keyword.lower() in doc.page_content.lower() else 0 for doc in docs]
            precision_scores.append(sum(key_rel)/len(docs))
        #     print(f"Keyword relevance: {key_rel}")
        #     print(f"Precision scores: {precision_scores}")
        #     print(f"docs: {len(docs)}")
        #     print(f"Keyword: {keyword.lower()} \n")
        # print("-"*50)
        scores += [sum(precision_scores) / len(test.keywords)]
    return sum(scores)/len(tests)


In [ ]:
calculate_precision(tests)

## Recall @ K
Recal is also a `predictive` metric that measures `coverage` of relevant items.

Recal answers the question "Of all the relevant items, how many were captured in the top K?"

Its is calculated by:
```python
    recall = number_of_relevant_items_at_K / total_number_of_relevant_items
```

Unike `Precision`, recall is also not a good predictive metric where number of relevant items is too large.

In cases where Relevant items is larger than K, Recall is `capped`. i.e.,
```python
    #if k = 10 while relevant items is 100, Highest Recall is capped at:
    Recall = 10/100
```
Recall works best where the number of relevant items is fairly low such as:
- Legal and Medical searching
- Topic-specific searches

Recall is `not` a good measure of coverage for the entire dataset especially where the number of relevant items varries per user or test question.

In [ ]:
#implement
#recall = relevant_docs / total_relevance
def calculate_recall(tests: TestQuestion, collection: Chroma._collection, k: int) ->float:
    scores = []
    for test in tests:
        all_docs = retriever.invoke(test.question, k=collection.count())
        total_relevance = []
        key_relevance = []
        recal_scores = []
        for keyword in test.keywords:
            key = keyword.lower()
            total_relevance = [1 if key in doc.page_content.lower() else 0 for doc in all_docs]
            key_relevance = total_relevance[:k]
            recal_scores.append(sum(key_relevance) / sum(total_relevance) if sum(total_relevance) > 0 else 0)
            # print(f"Keyword Relevance {key_relevance}")
            # print(f"Total Relevance {sum(total_relevance)}")
            # print(f"Keyword recal scores{recal_scores}")
            # print(f"Keyword {key} \n")
        scores += [sum(recal_scores) / len(test.keywords)]
        # print(f"Test score: {sum(recal_scores) / len(test.keywords)}")
        # print("-"*50)
    return f"Total Recall: {(sum(scores) / len(tests)):,.3f}"

In [ ]:
#implement
collection = Chroma(persist_directory="../vector_db/", embedding_function=embeddings)._collection

calculate_recall(tests, collection, 4)

## F-BETA SCORE @ K

This is the sweetspot for taking to account both `correctness` and `coverage` using both `Precision` and `Recall`. 

```python
    fbeta = ((1 + b**2) * precision * recall ) / ((b**2 * precision) + recall)
```
Beta in this case is used to set the importance of `recall` over `precision`.
By setting ```python beta = 1``` you get the `standard F1 score` which is a harmonic mean of precision and recall.

In [ ]:
#implement
#beta = 1 gives the harmonic F1 score
#beta > 1 prioritises recal
#beta < 1 prioritises precision because as beta approaches 0 the function approaches 1 + precision
#beta is a positive real valued number greater than 0

def calculate_f_beta(tests: TestQuestion, beta: int = 1) ->float:
    precision = calculate_precision(tests)
    recall = calculate_precision(tests)
    beta_score = ((1 + beta**2) * precision * recall) / ((beta**2 * precision) + recall)

    return beta_score
        

In [ ]:
#implement
calculate_f_beta(tests)